In [4]:
import pandas as pd
import polars as pl
import svy

In [5]:
h4_df = pd.read_csv('../data/processed/h4_analytic_sample.csv')
h4_pooled = h4_df.assign(wtmec2yr_pooled = h4_df['WTMEC2YR'] / 3)

design = svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='wtmec2yr_pooled')

# Primary model: excludes lipid-lowering medication users entirely
h4_primary = h4_pooled[~h4_pooled['on_lipid_lowering']]
sample_primary = svy.Sample(pl.from_pandas(h4_primary), design=design)

model_primary = svy.GLM(sample_primary)
model_primary.fit(
    y='LBXTC', 
    x=['LBXT3F', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='gaussian')
print(model_primary)


GLM: Gaussian (identity)
  Modeling : LBXTC
  n            : 6311          DF Residuals : 45
  Deviance     : 9.3060e+06    Scale        : 192356.3549
  AIC          : 63974.7185    BIC          : -
  R-squared    : 0.09841       R-sq (adj)   : 0.09784
  F-stat (adj) : 96.17791      Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.      Std.Err.  t         P>|t|   [0.025     0.975]   
  ------------  ---------  --------  --------  ------  ---------  ---------
  _intercept_   146.96740  7.92473   18.54541  <0.001  131.00618  162.92863
  LBXT3F        1.53209    1.93248   0.79281   0.4320  -2.36012   5.42430  
  RIDAGEYR      0.81510    0.04321   18.86176  <0.001  0.72806    0.90214  
  RIAGENDR_2.0  0.42385    1.35573   0.31263   0.7560  -2.30673   3.15442  
  BMXBMI        0.43221    0.10603   4.07614   <0.001  0.21865    0.64577  


/var/folders/rs/yth10b8d4v1043_hxt3sv4x40000gn/T/ipykernel_26870/1002399425.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  h4_pooled = h4_df.assign(wtmec2yr_pooled = h4_df['WTMEC2YR'] / 3)
